In [1]:
import pandas as pd
import numpy as np

In [3]:
FEATURES = [
    'Flow Duration',
    'Flow Packets/s',
    'Flow Bytes/s',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Fwd Packet Length Mean',
    'Fwd IAT Mean',
    'SYN Flag Count',
    'Packet Length Mean',
    'Packet Length Std',
    'Label'
]

df_udp = pd.read_csv('../data/raw/DrDos_UDP.csv', low_memory=False)
df_udp.columns = df_udp.columns.str.strip()
print(f'UDP Loaded: {df_udp.shape}')
print(f'UDP Labels: {df_udp['Label'].value_counts().to_dict()}')

UDP Loaded: (3136802, 88)
UDP Labels: {'DrDoS_UDP': 3134645, 'BENIGN': 2157}


In [5]:
df_syn = pd.read_csv('../data/raw/Syn.csv', low_memory=False)
df_syn.columns = df_syn.columns.str.strip()
print(f'SYN loaded: {df_syn.shape}')
print(f'SYN labels: {df_syn['Label'].value_counts().to_dict()}')

SYN loaded: (1582681, 88)
SYN labels: {'Syn': 1582289, 'BENIGN': 392}


In [7]:
available_upd = [f for f in FEATURES if f in df_udp.columns]
available_syn = [f for f in FEATURES if f in df_syn.columns]

print(f'Available in UDP: {available_upd}')
print(f'Missing is UDP: {[f for f in FEATURES if f not in df_udp.columns]}')
print(f'Available in SYN: {available_syn}')
print(f'Missing in SYN: {[f for f in FEATURES if f not in df_syn.columns]}')

Available in UDP: ['Flow Duration', 'Flow Packets/s', 'Flow Bytes/s', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packet Length Mean', 'Fwd IAT Mean', 'SYN Flag Count', 'Packet Length Mean', 'Packet Length Std', 'Label']
Missing is UDP: []
Available in SYN: ['Flow Duration', 'Flow Packets/s', 'Flow Bytes/s', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packet Length Mean', 'Fwd IAT Mean', 'SYN Flag Count', 'Packet Length Mean', 'Packet Length Std', 'Label']
Missing in SYN: []


In [9]:
df_udp = df_udp[available_upd].copy()
df_syn = df_syn[available_syn].copy()

df_udp['Attack_type'] = df_udp['Label'].apply(
    lambda x: 'UDP_Flood' if x != 'BENIGN' else 'BENIGN'
)

df_syn['Attack_type'] = df_syn['Label'].apply(
    lambda x: 'SYN_Flood' if x != 'BENIGN' else 'BENIGN'
)

print(f"UDP subset shape: {df_udp.shape}")
print(f"SYN subset shape: {df_syn.shape}")

UDP subset shape: (3136802, 12)
SYN subset shape: (1582681, 12)


In [14]:
df = pd.concat([df_udp, df_syn], ignore_index=True)
print('Final Dataset Shape:', df.shape)

df['binary_label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)

print('\n Label Distribution:')
print(df['binary_label'].value_counts())
print('Attack type:')
print(df['Attack_type'].value_counts())

Final Dataset Shape: (4719483, 12)

 Label Distribution:
binary_label
1    4716934
0       2549
Name: count, dtype: int64
Attack type:
Attack_type
UDP_Flood    3134645
SYN_Flood    1582289
BENIGN          2549
Name: count, dtype: int64


In [18]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print('NaN counts per column before cleaning:')
print(df.isnull().sum())

df.dropna(inplace=True)

print(f'\n Shape after cleaning: {df.shape}')
print(f'Total NaN remaining: {df.isnull().sum().sum()}')

NaN counts per column before cleaning:
Flow Duration             0
Flow Packets/s            0
Flow Bytes/s              0
Total Fwd Packets         0
Total Backward Packets    0
Fwd Packet Length Mean    0
Fwd IAT Mean              0
SYN Flag Count            0
Packet Length Mean        0
Packet Length Std         0
Label                     0
Attack_type               0
binary_label              0
dtype: int64

 Shape after cleaning: (4476493, 13)
Total NaN remaining: 0


In [22]:
benign = df[df['binary_label'] == 0]
udp_attack = df[df['Attack_type'] == 'UDP_Flood']
syn_attack = df[df['Attack_type'] == 'SYN_Flood']

print(f'Total BENIGN rows available: {len(benign)}')
print(f'Total UDP Flood rows available: {len(udp_attack)}')
print(f'Total SYN Flood rows available: {len(syn_attack)}')

ATTACK_SAMPLE = 30000
udp_sample = udp_attack.sample(n=min(ATTACK_SAMPLE, len(udp_attack)), random_state=42)
syn_sample = syn_attack.sample(n=min(ATTACK_SAMPLE, len(syn_attack)), random_state=42)

df_final = pd.concat([benign, udp_sample, syn_sample], ignore_index=True)
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Final dataset shape: {df_final.shape}')
print('Final Label Distribution:')
print(df_final['binary_label'].value_counts())
print('\nFinal attack type:')
print(df_final['Attack_type'].value_counts())

Total BENIGN rows available: 2508
Total UDP Flood rows available: 3094002
Total SYN Flood rows available: 1379983
Final dataset shape: (62508, 13)
Final Label Distribution:
binary_label
1    60000
0     2508
Name: count, dtype: int64

Final attack type:
Attack_type
UDP_Flood    30000
SYN_Flood    30000
BENIGN        2508
Name: count, dtype: int64


In [30]:
import joblib
candidate_cols = [
    col for col in df_final.columns if col not in ['Label', 'Attack_type', 'binary_label']
]

corr = (df_final[candidate_cols].corrwith(df_final['binary_label']).abs().sort_values(ascending=False))

print('Correlation with binary label: ', corr)

final_features = corr.head(10).index.to_list()

df_final[final_features]

print('\nFinal input features for model:')
for i, f in enumerate(final_features, 1):
  print(f' {i}. {f}')

print(f'\nTotal selected features: {len(final_features)}')
print(f'Total rows: {len(df_final)}')
print(f"BENIGN: {len(df_final[df_final['binary_label'] == 0])}")
print(f"ATTACK (UDP + SYN): {len(df_final[df_final['binary_label'] == 1])}")

print('\nBasic statistics (Selected Features):')
print(df_final[final_features].describe().round(2))


Correlation with binary label:  Packet Length Std         0.336889
Total Backward Packets    0.228233
Flow Packets/s            0.180344
Fwd Packet Length Mean    0.150311
Packet Length Mean        0.120773
Total Fwd Packets         0.109365
Flow Bytes/s              0.100140
SYN Flag Count            0.062312
Fwd IAT Mean              0.019317
Flow Duration             0.018606
dtype: float64

Final input features for model:
 1. Packet Length Std
 2. Total Backward Packets
 3. Flow Packets/s
 4. Fwd Packet Length Mean
 5. Packet Length Mean
 6. Total Fwd Packets
 7. Flow Bytes/s
 8. SYN Flag Count
 9. Fwd IAT Mean
 10. Flow Duration

Total selected features: 10
Total rows: 62508
BENIGN: 2508
ATTACK (UDP + SYN): 60000

Basic statistics (Selected Features):
       Packet Length Std  Total Backward Packets  Flow Packets/s  \
count           62508.00                62508.00        62508.00   
mean               10.56                    0.38      1015490.31   
std                45.15     

In [33]:
df_final.to_csv('../data/processed/clean_dataset.csv', index=False)

print('Cleaned dataset saved!')
print(f'Total rows: {len(df_final)}')
print(f'Features: {candidate_cols}')

Cleaned dataset saved!
Total rows: 62508
Features: ['Flow Duration', 'Flow Packets/s', 'Flow Bytes/s', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packet Length Mean', 'Fwd IAT Mean', 'SYN Flag Count', 'Packet Length Mean', 'Packet Length Std']
